In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import os
import subprocess
from pathlib import Path

import anndata as ad
import colorcet as cc
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
from scipy.sparse import load_npz
from spatialleiden import distance2connectivity, spatialleiden

from utils import (
    DATA_DIR,
    RESULT_DIR,
    SHELL,
    activate_env_cmd,
    mm,
    remove_tick_and_label,
)

In [ ]:
abc_dir = DATA_DIR / "ABC_atlas"
input = abc_dir / "Zhuang-ABCA-1.h5ad"
ccf_annot_file = abc_dir / "CCF.tsv"

results_abc = RESULT_DIR / "ABC_atlas"
processed_h5ad = results_abc / "Zhuang-ABCA-1.h5ad"

script_dir = Path("batch_integration")

seed = 42
n_components = 50
# n_neighbors = 25

In [ ]:
results_abc.mkdir(exist_ok=True, parents=True)

In [ ]:
env = os.environ.copy()
env["MPLBACKEND"] = "pdf"  # avoid matplotlib backend issues

# Preprocess

In [ ]:
adata = ad.read_h5ad(input)

layer = "scaled"
adata.layers[layer] = adata.X.copy()
adata.X = adata.X.astype("int16")

sc.pp.log1p(adata, layer=layer)
sc.pp.normalize_total(adata, layer=layer)
sc.pp.scale(adata, zero_center=False, layer=layer)
sc.pp.pca(adata, n_comps=n_components, zero_center=True, random_state=seed, layer=layer)


sq.gr.spatial_neighbors(
    adata, library_key=adata.uns["batch_keys"][0], coord_type="generic", n_neighs=10
)

adata.obsp["spatial_connectivities"] = distance2connectivity(
    adata.obsp["spatial_distances"]
)

adata.write_h5ad(processed_h5ad)

# Batch correction

BBKNN selects *n* neighbors per batch, given that we have 147 sections this approach seems less suitable.

In [ ]:
methods = [
    "no_batch_integration",
    "harmony",
    "scvi",
]

In [ ]:
for method in methods:
    out_file = results_abc / f"{method}.npz"
    script_path = script_dir / f"{method}.py"

    conda_cmd = activate_env_cmd(method)
    cmd = f"{script_path} {processed_h5ad} {out_file}"

    # use same number of components for scVI as PCs for harmony
    if method == "scvi":
        cmd += f" --n_comp {n_components}"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Clustering

In [ ]:
import time

In [ ]:
adata = ad.read_h5ad(processed_h5ad)

In [ ]:
for file in results_abc.glob("*.npz"):
    method = file.stem

    print(f"Processing {method}")

    adata.obsp["connectivities"] = load_npz(file)

    start = time.time()
    spatialleiden(
        adata,
        resolution=(1, 1),
        directed=(False, True),
        layer_ratio=2.5,
        n_iterations=3,
    )
    end = time.time()
    print(f"Time: {(end - start) / 60:.2f} min")

    columns = ["spatialleiden"] + list(adata.uns["batch_keys"])

    labels = adata.obs[columns]
    labels.loc[:, ["x", "y", "z"]] = adata.obsm["spatial"]
    labels.to_parquet(results_abc / f"{method}.parquet")

# Plotting

In [ ]:
scatter_kwargs = {
    "s": 1,
    "linewidth": 0,
}

In [ ]:
def plot_sections(label_file: str | Path):
    labels = pd.read_parquet(label_file).astype({"spatialleiden": "category"})

    g = sns.FacetGrid(labels, col="brain_section_label", col_wrap=7, legend_out=False)
    _ = g.map_dataframe(
        sns.scatterplot,
        x="x",
        y="y",
        hue="spatialleiden",
        palette=sns.color_palette(
            cc.glasbey, n_colors=labels["spatialleiden"].cat.categories.size
        ),
        legend=False,
        **scatter_kwargs,
    )

    for ax in g.axes.flat:
        ax.set(aspect=1)
        remove_tick_and_label(ax)

    return g

In [ ]:
# Don't show figure
_ = plt.ioff()

## No integration

In [ ]:
fig = plot_sections(results_abc / "no_batch_integration.parquet")
fig.savefig(results_abc / "no_batch_integration.png")

## Harmony

In [ ]:
fig = plot_sections(results_abc / "harmony.parquet")
fig.savefig(results_abc / "harmony.png")

## scVI

In [ ]:
fig = plot_sections(results_abc / "scvi.parquet")
fig.savefig(results_abc / "scvi.png")

## GIFs

In [ ]:
conda_cmd = activate_env_cmd("gif")
script_path = Path("gif") / "gif.py"

In [ ]:
batch_key = "brain_section_label"
fps = 6
height = 6.5

for file in results_abc.glob("*.parquet"):
    out_file = results_abc / f"{file.stem}.gif"
    cmd = (
        f"{script_path} {file} {out_file} "
        f"--batch {batch_key} "
        f"--fps {fps} "
        f"--height {height} "
    )

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL, env=env)

# Evaluation

In [ ]:
import numpy as np


def domain_enrichment(
    label_df: pd.DataFrame,
    anno_df: pd.DataFrame,
    anno_key: str,
    domain_key: str = "spatialleiden",
):
    counts = (
        label_df.join(anno_df, how="inner")
        .groupby([domain_key, anno_key])
        .size()
        .to_frame("n")
    )

    densities = counts.assign(
        domain_density=lambda df: df["n"]
        / df["n"].groupby(domain_key).transform("sum"),
        density=lambda df: df["n"].groupby(anno_key).transform("sum") / df["n"].sum(),
        FC=lambda df: df["domain_density"] / df["density"],
        log_FC=lambda df: np.log2(df["FC"]),
    ).reset_index()

    return densities


def plot_enrichment(
    df: pd.DataFrame, key: str, value_key: str = "FC", index_key: str = "spatialleiden"
):
    matrix = df.pivot(index=index_key, columns=key, values=value_key)
    return sns.clustermap(
        matrix, row_cluster=False, vmax=10, cmap="rocket_r", xticklabels=False
    )  # .fillna(1)


def plot_cluster_size(df, key: str = "spatialleiden"):
    fig, ax = plt.subplots()
    sns.histplot(df[key], discrete=True, ax=ax)
    ax.set(ylabel="# cells", xlabel=f"{key} cluster")
    # ax.set_yticklabels(ax.get_yticklabels(), rotation=45)

In [ ]:
ccf = pd.read_table(ccf_annot_file, index_col="cell_label")
ccf = ccf.astype({c: "category" for c in ccf.columns[3:]})

celltypes = ad.read_h5ad(input).obs.drop(columns=["brain_section_label"])

In [ ]:
scatter_kwargs = {"s": 0.05, "linewidth": 0}
hm_kwargs = dict(
    vmin=0, vmax=10, clip_on=False, cmap="rocket_r", xticklabels=False, yticklabels=True
)
cbar_kwargs = dict(label="Enrichment Score")

In [ ]:
for label_file in results_abc.glob("*.parquet"):
    method = label_file.stem
    label = pd.read_parquet(label_file).astype({"spatialleiden": "category"})

    plot_cluster_size(label)

    g = plot_enrichment(domain_enrichment(label, celltypes, "subclass"), "subclass")
    clustered_celltypes = g.data2d
    plt.close()

    f = plot_enrichment(
        domain_enrichment(label, ccf, "parcellation_index"), "parcellation_index"
    )
    clustered_ccf = f.data2d
    plt.close()

    keep = [15, 30, 45, 60, 74, 90, 105, 120, 135]

    sections = label["brain_section_label"].cat.categories
    sections = (
        sections.str.extract("(\\d+)$")
        .astype(int)
        .set_index(sections)
        .rename(columns={0: "section"})
    )

    label = (
        pd.merge(label, sections, left_on="brain_section_label", right_index=True)
        .loc[lambda df: df["section"].isin(keep)]
        .assign(section=lambda df: df["section"].astype("category"))
    )

    import matplotlib.lines as mlines

    palette = dict(
        zip(
            label["spatialleiden"].cat.categories,
            sns.color_palette(
                cc.glasbey, n_colors=label["spatialleiden"].cat.categories.size
            ),
        )
    )

    fig = plt.figure(
        figsize=(170 * mm, 120 * mm),
    )

    figs = fig.subfigures(nrows=2, height_ratios=[35, 85], hspace=0)

    # plot sections
    axs = figs[0].subplots(
        ncols=label["section"].cat.categories.size,
        sharex=True,
        sharey=True,
        gridspec_kw={"wspace": 0.1},
    )
    figs[0].suptitle("Section")
    figs[0].subplots_adjust(left=0, right=1, top=0.85, bottom=0)

    for (n, df), ax in zip(label.groupby("section"), axs):
        df[["x", "y"]] *= -1
        sns.scatterplot(
            data=df,
            x="x",
            y="y",
            hue="spatialleiden",
            palette=palette,
            legend=False,
            ax=ax,
            **scatter_kwargs,
        )
        ax.set(aspect=1)
        ax.set_title(n, size="medium")
        remove_tick_and_label(ax)

    handles = [
        mlines.Line2D(
            [], [], color=color, marker="o", linestyle="None", markersize=5, label=label
        )
        for label, color in palette.items()
    ]
    figs[0].legend(
        handles=handles,
        loc="lower center",
        ncols=len(palette) / 2,
        fontsize="small",
        frameon=False,
        handletextpad=0.3,
        columnspacing=1,
        bbox_to_anchor=(0.5, -0.25),
    )

    # plot heatmaps
    axs = figs[1].subplots(
        ncols=3, width_ratios=[1, 1, 0.05], gridspec_kw={"wspace": 0.1, "hspace": 0}
    )

    sns.heatmap(
        clustered_celltypes,
        ax=axs[0],
        cbar_ax=axs[2],
        cbar_kws=cbar_kwargs,
        **hm_kwargs,
    )
    sns.heatmap(clustered_ccf, ax=axs[1], cbar=False, **hm_kwargs)

    axs[0].set(ylabel="SpatialLeiden cluster")
    axs[0].set_yticklabels(axs[0].get_yticklabels(), fontsize="small")
    axs[1].yaxis.set_visible(False)
    axs[2].tick_params(labelsize="small")

    for ax in axs:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)

    fig.savefig(results_abc / f"fig_{method}.png", dpi=300)  # , bbox_inches="tight"
    break

In [ ]:
for label_file in results_abc.glob("*.parquet"):
    method = label_file.stem
    label = pd.read_parquet(label_file).astype({"spatialleiden": "category"})

    plot_cluster_size(label)

    g = plot_enrichment(domain_enrichment(label, celltypes, "subclass"), "subclass")
    clustered_celltypes = g.data2d
    plt.close()

    f = plot_enrichment(
        domain_enrichment(label, ccf, "parcellation_index"), "parcellation_index"
    )
    clustered_ccf = f.data2d
    plt.close()

    keep = list(range(10, 150, 10))

    sections = label["brain_section_label"].cat.categories
    sections = (
        sections.str.extract("(\\d+)$")
        .astype(int)
        .set_index(sections)
        .rename(columns={0: "section"})
    )

    label = (
        pd.merge(label, sections, left_on="brain_section_label", right_index=True)
        .loc[lambda df: df["section"].isin(keep)]
        .assign(section=lambda df: df["section"].astype("category"))
    )

    import matplotlib.lines as mlines

    palette = dict(
        zip(
            label["spatialleiden"].cat.categories,
            sns.color_palette(
                cc.glasbey, n_colors=label["spatialleiden"].cat.categories.size
            ),
        )
    )

    fig = plt.figure(
        figsize=(170 * mm, 160 * mm),
    )

    figs = fig.subfigures(nrows=2, height_ratios=[75, 85], hspace=0)

    # plot sections
    axs = figs[0].subplots(
        nrows=2,
        ncols=label["section"].cat.categories.size // 2,
        sharex=True,
        sharey=True,
        gridspec_kw={"wspace": 0.1, "hspace": 0.05},
    )
    figs[0].suptitle("Section")
    figs[0].subplots_adjust(left=0, right=1, top=0.85, bottom=0)

    for i, (n, df) in enumerate(label.groupby("section")):
        ax = axs[i // 7, i % 7]
        df[["x", "y"]] *= -1
        sns.scatterplot(
            data=df,
            x="x",
            y="y",
            hue="spatialleiden",
            palette=palette,
            legend=False,
            ax=ax,
            **scatter_kwargs,
        )
        ax.set(aspect=1)
        ax.set_title(n, size="medium")
        remove_tick_and_label(ax)

    handles = [
        mlines.Line2D(
            [], [], color=color, marker="o", linestyle="None", markersize=5, label=label
        )
        for label, color in palette.items()
    ]
    figs[0].legend(
        handles=handles,
        loc="lower center",
        ncols=len(palette) // 2,
        fontsize="small",
        frameon=False,
        handletextpad=0.3,
        columnspacing=1,
        bbox_to_anchor=(0.5, -0.15),
    )

    # plot heatmaps
    axs = figs[1].subplots(
        ncols=3, width_ratios=[1, 1, 0.05], gridspec_kw={"wspace": 0.1, "hspace": 0}
    )

    sns.heatmap(
        clustered_celltypes,
        ax=axs[0],
        cbar_ax=axs[2],
        cbar_kws=cbar_kwargs,
        **hm_kwargs,
    )
    sns.heatmap(clustered_ccf, ax=axs[1], cbar=False, **hm_kwargs)

    axs[0].set(ylabel="SpatialLeiden cluster")
    axs[0].set_yticklabels(axs[0].get_yticklabels(), fontsize="small")
    axs[1].yaxis.set_visible(False)
    axs[2].tick_params(labelsize="small")

    for ax in axs:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1)

    fig.savefig(results_abc / f"fig_{method}.png", dpi=300, bbox_inches="tight")